<div style="background:linear-gradient(135deg, rgba(249,112,102,0.18), rgba(249,112,102,0.02)); border-left:6px solid #F97066; border-radius:10px; padding:20px 24px; margin-bottom:20px;">
<h1 style="margin:0; color:#F97066; font-size:1.8em;">🤖 Fundamentos de Agentes de IA</h1>
<p style="margin:6px 0 0; opacity:0.8;">Unidad 5 — Del LLM que responde al agente que razona, actúa y observa</p>
</div>

Este notebook presenta los fundamentos de los agentes de IA (*agentic AI*): qué distingue a un agente de una simple llamada a un LLM, el patrón **ReAct** (razonar y actuar en ciclos), y cómo construir un agente básico con LangChain y Google Gemini. Los temas más avanzados — memoria, múltiples herramientas, control explícito del flujo, RAG, sistemas multiagente y evaluación — se profundizan cada uno en su propio notebook, indicado al final de este.

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0 24px; background:rgba(14,165,233,0.04);">
<strong>📑 Contenido de esta guía</strong>
<ol style="margin:8px 0 0; padding-left:20px;">
<li><a href="#que-es-un-agente">¿Qué es un agente de IA?</a></li>
<li><a href="#patron-react">El patrón ReAct</a></li>
<li><a href="#construccion-agente">Construcción de un agente con LangChain</a></li>
<li><a href="#cierre">Cierre y próximos pasos</a></li>
</ol>
</div>

<a id="que-es-un-agente"></a>

## <span style="color:#F97066;">¿Qué es un agente de IA?</span>

Un LLM, por sí solo, solo puede responder con lo que "sabe" a partir de su entrenamiento: recibe texto y produce texto. No puede consultar información actualizada, hacer cálculos exactos y confiables, ni ejecutar ninguna acción sobre el mundo — para eso, un **agente** le agrega al LLM tres elementos:

- **Herramientas** (*tools*): funciones concretas que el agente puede invocar para obtener información o ejecutar una acción (una calculadora, una búsqueda, una consulta a una base de datos, el envío de un correo).
- **Un ciclo de decisión**: en lugar de responder de una sola vez, el agente decide en cada paso si ya tiene suficiente información para responder, o si necesita usar una herramienta para conseguir más — y repite ese ciclo hasta resolver la tarea.
- **Autonomía**: el LLM mismo decide qué herramienta usar, con qué argumentos, y cuándo detenerse — nadie programó de antemano la secuencia exacta de pasos para cada consulta posible.

Esta combinación es lo que permite que un agente resuelva tareas que un LLM aislado no podría resolver de forma confiable.

<a id="patron-react"></a>

## <span style="color:#F97066;">El patrón ReAct</span>

**ReAct** (*Reason + Act*, Yao et al., 2022) es uno de los patrones más usados para estructurar el ciclo de decisión de un agente. En cada iteración, el LLM produce explícitamente:

1. **Thought** (razonamiento): un paso de razonamiento en lenguaje natural sobre qué hacer a continuación.
2. **Action** (acción): el nombre de una herramienta a invocar y sus argumentos.
3. **Observation** (observación): el resultado que devolvió esa herramienta, que se agrega de vuelta a lo que el LLM "ve" antes de decidir el siguiente paso.

Este ciclo Thought → Action → Observation se repite hasta que el LLM decide que ya tiene toda la información necesaria, momento en el que produce una **Final Answer** en lugar de una nueva acción. Intercalar explícitamente el razonamiento entre cada acción (en vez de solo encadenar acciones) hace que el agente sea más confiable: puede replantear su estrategia si una herramienta no devolvió lo esperado, en lugar de continuar ciegamente.

<a id="construccion-agente"></a>

## <span style="color:#F97066;">Construcción de un agente con LangChain</span>

<span style="color:#F97066;">1.</span> Configuración del modelo

Se usa Gemini de Google a través de `langchain-google-genai`. La clave de API se carga desde un archivo `.env` (variable `GOOGLE_API_KEY`) que nunca debe subirse al repositorio.

In [1]:
import os
import logging
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Cargar variables de entorno (requiere un archivo .env con GOOGLE_API_KEY=tu_clave)
load_dotenv()

# Silenciar un aviso benigno de la librería sobre el uso de function calling automático
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

# 2. Configurar el LLM (Gemini). Temperatura baja para un razonamiento más determinista.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    api_key=os.getenv("GOOGLE_API_KEY"),
)

print("LLM configurado correctamente.")

LLM configurado correctamente.


<span style="color:#F97066;">2.</span> Definir una herramienta

Una herramienta es, simplemente, una función de Python decorada con `@tool`: LangChain usa su nombre, sus anotaciones de tipo y su docstring para describírsela al LLM, que decide cuándo y con qué argumentos invocarla.

<div style="border-left:4px solid #6366F1; background:rgba(99,102,241,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>📝 Nota</strong><br>
El docstring de una herramienta no es un simple comentario: es la única descripción que el LLM recibe de lo que hace la función. Un docstring vago o incompleto es una causa frecuente de que un agente elija mal la herramienta o la use con argumentos incorrectos.
</div>

In [2]:
import math
from langchain.tools import tool


@tool
def calcular_area_circulo(radio: float) -> str:
    """Calcula el área de un círculo dado su radio.

    Args:
        radio: El radio del círculo, en las unidades que sean.

    Returns:
        El área calculada, con una breve explicación.
    """
    area = math.pi * radio ** 2
    return f"El área del círculo con radio {radio} es aproximadamente {area:.2f} unidades cuadradas."


print("Herramienta definida correctamente.")

Herramienta definida correctamente.


<span style="color:#F97066;">3.</span> Construir el agente

<code>create_agent</code> (LangChain 1.0) construye, internamente, un grafo de LangGraph que implementa el ciclo ReAct: un nodo que llama al LLM y otro que ejecuta la herramienta solicitada, conectados en un ciclo que se repite hasta que el LLM responde sin pedir ninguna acción adicional. La sección <code>4-grafos-langgraph.ipynb</code> muestra cómo construir ese mismo grafo explícitamente, nodo por nodo.

In [3]:
from langchain.agents import create_agent

# 1. Crear el agente: LLM + herramientas disponibles + instrucciones generales
agente = create_agent(
    model=llm,
    tools=[calcular_area_circulo],
    system_prompt=(
        "Eres un asistente que razona paso a paso y usa las herramientas disponibles "
        "cuando el cálculo lo requiere, en lugar de calcular de memoria."
    ),
)

print("Agente ReAct construido correctamente.")

Agente ReAct construido correctamente.


<span style="color:#F97066;">4.</span> Ejecutar el agente

Un agente construido con <code>create_agent</code> recibe y devuelve una lista de <code>messages</code> (la misma convención de LangGraph): se le pasa la pregunta como un mensaje de tipo <code>"user"</code>, y su respuesta final es el último mensaje de la lista devuelta.

In [4]:
consulta = "Calcula el área de un círculo de radio 5 y explica por qué usas esa fórmula."

# 1. Invocar el agente con la consulta como un mensaje de usuario
resultado = agente.invoke({"messages": [{"role": "user", "content": consulta}]})

# 2. La respuesta final es el último mensaje de la conversación
print("Respuesta final del agente:\n")
print(resultado["messages"][-1].text)

Respuesta final del agente:

Para calcular el área de un círculo, utilizamos la siguiente fórmula matemática:

$$\text{Área} = \pi \cdot r^2$$

### ¿Por qué usamos esta fórmula?

1. **El concepto de área:** El área mide la superficie bidimensional encerrada dentro de una circunferencia.
2. **El radio ($r$):** Es la distancia desde el centro del círculo hasta cualquier punto de su borde. La fórmula se basa en el radio porque todas las dimensiones del círculo dependen directamente de él.
3. **El número $\pi$ (Pi):** Es una constante matemática fundamental que representa la relación entre la longitud de la circunferencia (su perímetro) y su diámetro. Siempre es el mismo valor aproximado ($3.14159...$).
4. **¿Por qué está al cuadrado ($r^2$)?** Geométricamente, el área se mide en unidades cuadradas. Al multiplicar el radio por sí mismo ($r \times r$), obtenemos una escala proporcional al espacio bidimensional que ocupa el círculo.

---

### Aplicando la fórmula para un radio de $5$:

1. Su

<span style="color:#F97066;">5.</span> Qué ocurrió internamente

Se puede inspeccionar la lista completa de mensajes para ver el ciclo ReAct paso a paso: el mensaje del usuario, la decisión del LLM de invocar <code>calcular_area_circulo</code> (con sus argumentos), el resultado que devolvió la herramienta, y finalmente la respuesta sintetizada por el LLM.

In [5]:
for mensaje in resultado["messages"]:
    tipo = mensaje.__class__.__name__
    contenido = mensaje.text if mensaje.text else getattr(mensaje, "tool_calls", "")
    print(f"[{tipo}] {contenido}")

[HumanMessage] Calcula el área de un círculo de radio 5 y explica por qué usas esa fórmula.
[AIMessage] [{'name': 'calcular_area_circulo', 'args': {'radio': 5}, 'id': 'call_309775', 'type': 'tool_call'}]
[ToolMessage] El área del círculo con radio 5.0 es aproximadamente 78.54 unidades cuadradas.
[AIMessage] Para calcular el área de un círculo, utilizamos la siguiente fórmula matemática:

$$\text{Área} = \pi \cdot r^2$$

### ¿Por qué usamos esta fórmula?

1. **El concepto de área:** El área mide la superficie bidimensional encerrada dentro de una circunferencia.
2. **El radio ($r$):** Es la distancia desde el centro del círculo hasta cualquier punto de su borde. La fórmula se basa en el radio porque todas las dimensiones del círculo dependen directamente de él.
3. **El número $\pi$ (Pi):** Es una constante matemática fundamental que representa la relación entre la longitud de la circunferencia (su perímetro) y su diámetro. Siempre es el mismo valor aproximado ($3.14159...$).
4. **¿Por q

---

<a id="cierre"></a>

# <span style="color:#F97066;">🎯 Cierre y próximos pasos</span>

<div style="border-left:4px solid #14B8A6; background:rgba(20,184,166,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>✅ Resumen</strong><br>
Este notebook presentó los fundamentos de los agentes de IA:

- Qué agrega un agente a un LLM aislado: herramientas, un ciclo de decisión y autonomía.
- El patrón ReAct: Thought → Action → Observation, repetido hasta llegar a una Final Answer.
- Cómo construir y ejecutar un agente con `create_agent` de LangChain 1.0, y cómo inspeccionar su ciclo de mensajes paso a paso.

</div>

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0; background:rgba(14,165,233,0.04);">
<strong>🔎 Profundice en cada tema</strong>
<p style="margin:8px 0;">Un solo ejemplo con una sola herramienta alcanza para ver el ciclo ReAct con claridad, pero un agente real casi siempre necesita recordar el contexto de la conversación, elegir entre varias herramientas, y a veces coordinarse con otros agentes. Los siguientes notebooks profundizan cada uno de esos temas:</p>
<ul style="margin:8px 0 0; padding-left:20px;">
<li><code>2-memoria-y-estado.ipynb</code> — memoria persistente entre turnos con el <code>checkpointer</code> de LangGraph.</li>
<li><code>3-multiples-herramientas.ipynb</code> — un agente con varias herramientas distintas, y cómo elige cuál usar.</li>
<li><code>4-grafos-langgraph.ipynb</code> — construir el grafo del agente a mano (nodos, aristas condicionales) en vez de la caja negra de <code>create_agent</code>.</li>
<li><code>5-rag-como-herramienta.ipynb</code> — recuperación de información propia (RAG) integrada como una herramienta más del agente.</li>
<li><code>6-multiagentes.ipynb</code> — varios agentes especializados coordinados por un agente supervisor.</li>
<li><code>7-evaluacion-de-agentes.ipynb</code> — cómo medir si un agente elige bien sus herramientas y responde correctamente.</li>
</ul>
</div>

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0; background:rgba(14,165,233,0.04);">
<strong>➡️ Continúe con</strong>
<ul style="margin:8px 0 0; padding-left:20px;">
<li><code>2-memoria-y-estado.ipynb</code> — el primer paso para profundizar en lo visto en este notebook.</li>
</ul>
</div>